In [ ]:
import numpy as np
import pandas as pd
import os

# Just confirm dataset is mounted — don't list every file
print("Input files:")
for folder in os.listdir('/kaggle/input'):
    print(f"  /kaggle/input/{folder}")

In [ ]:
# ── CELL 2: Config ─────────────────────────────────────────────────────────
import os
import numpy as np

DATA_DIR    = "/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset"
CACHE_DIR   = "/kaggle/working/cache"
RESULTS_DIR = "/kaggle/working/results"

os.makedirs(CACHE_DIR,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

CLASS_NAMES  = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]
CLASS_COUNTS = {0: 3616, 1: 6012, 2: 10192, 3: 1345}
N_CLASSES    = 4
client_names = ["A", "B", "C"]

TEST_RATIO       = 0.15
DIRICHLET_ALPHA  = 0.5

CLIENT_DOMINANCE = {
    0: {"class": 0, "ratio": 0.70},  # Client A — 70% COVID
    1: {"class": 1, "ratio": 0.70},  # Client B — 70% Lung Opacity
    2: {"class": 3, "ratio": 0.70},  # Client C — 70% Viral Pneumonia
}

# Feature extraction config
FEATURE_DIM = 1280
BATCH_SIZE  = 128   # larger batch for faster extraction

print("Config loaded.")
print(f"  Classes     : {CLASS_NAMES}")
print(f"  Test ratio  : {TEST_RATIO}")
print(f"  Feature dim : {FEATURE_DIM}")

In [ ]:
# ── CELL 3: Imports ────────────────────────────────────────────────────────
from collections import Counter
from sklearn.model_selection import train_test_split

print("Imports done.")

In [ ]:
# ── CELL 4: Verify dataset structure ──────────────────────────────────────
# Run this to confirm the dataset is attached correctly
# If you see 'FileNotFoundError', go to Data tab → Add Data →
# search 'COVID-19 Radiography Database' by tawsifurrahman → Add

print("Scanning dataset root...\n")
for folder in sorted(os.listdir(DATA_DIR)):
    path = os.path.join(DATA_DIR, folder)
    if not os.path.isdir(path):
        continue
    # Dataset has images inside a /images subfolder
    img_path = os.path.join(path, "images")
    target   = img_path if os.path.exists(img_path) else path
    count    = len([f for f in os.listdir(target) if f.endswith(".png")])
    expected = CLASS_COUNTS.get(CLASS_NAMES.index(folder), "?") \
               if folder in CLASS_NAMES else "(unexpected folder)"
    status   = "OK" if count > 0 else "EMPTY"
    print(f"  [{status}] {folder:25s}: {count:6d} images   expected: {expected}")

In [ ]:
# ── CELL 5: Build all_paths + all_labels ──────────────────────────────────
all_paths  = []
all_labels = []

for label, class_name in enumerate(CLASS_NAMES):
    class_dir  = os.path.join(DATA_DIR, class_name)
    img_subdir = os.path.join(class_dir, "images")
    search_dir = img_subdir if os.path.isdir(img_subdir) else class_dir

    files = sorted([f for f in os.listdir(search_dir) if f.lower().endswith(".png")])

    for fname in files:
        all_paths.append(os.path.join(search_dir, fname))
        all_labels.append(label)

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels, dtype=np.int64)

print(f"Total images indexed: {len(all_paths)}")
print()
for label, class_name in enumerate(CLASS_NAMES):
    n = (all_labels == label).sum()
    print(f"  {class_name:25s}: {n:6d}")

In [ ]:
# ── CELL 6: Train / test split (stratified, 15% test) ─────────────────────
# IMPORTANT: split BEFORE partitioning into clients
# Test set is global and never seen by any client during training

train_indices = []
test_indices  = []

for label in range(N_CLASSES):
    class_idx = np.where(all_labels == label)[0]
    tr_idx, te_idx = train_test_split(
        class_idx,
        test_size    = TEST_RATIO,
        random_state = RANDOM_SEED,
        shuffle      = True
    )
    train_indices.extend(tr_idx.tolist())
    test_indices.extend(te_idx.tolist())

train_indices = np.array(train_indices)
test_indices  = np.array(test_indices)

print("Train / test split (stratified):")
print(f"  Total train : {len(train_indices):6d}")
print(f"  Total test  : {len(test_indices):6d}")
print()
for label, class_name in enumerate(CLASS_NAMES):
    tr = (all_labels[train_indices] == label).sum()
    te = (all_labels[test_indices]  == label).sum()
    print(f"  {class_name:25s}: {tr:5d} train  |  {te:4d} test")

In [ ]:
# ── CELL 7: Build per-class pools from TRAIN INDICES ONLY ─────────────────
# Partition draws only from training data — test set is never touched

class_indices = {
    label: train_indices[np.where(all_labels[train_indices] == label)[0]].copy()
    for label in range(N_CLASSES)
}

# Shuffle each pool independently
for label in class_indices:
    np.random.shuffle(class_indices[label])

# Viral Pneumonia is the smallest class — Client C is capped so it doesn't
# starve the other clients of VP images
# Cap = (VP train count) / 0.70 → keeps 70% dominance ratio
vp_train_count       = len(class_indices[3])
client_c_max         = int(vp_train_count / 0.70)

print(f"Train pool sizes:")
for label, class_name in enumerate(CLASS_NAMES):
    print(f"  {class_name:25s}: {len(class_indices[label])} images")
print()
print(f"Viral Pneumonia in train : {vp_train_count}")
print(f"Client C max samples     : {client_c_max}   (VP / 0.70)")

In [ ]:
# ── CELL 8: constrained_partition() ───────────────────────────────────────
# Strategy:
#   1. Client C first — takes ALL VP (dominant) + fills rest via Dirichlet
#   2. Client A — takes ALL remaining COVID (dominant) + fills rest
#   3. Client B — takes ALL remaining Lung Opacity (dominant) + fills rest
#   4. Any leftover images distributed proportionally across clients
#      so NOTHING is wasted

def constrained_partition(class_indices, client_dominance,
                           client_c_max, n_classes, dirichlet_alpha):
    pools    = {k: list(v) for k, v in class_indices.items()}
    assigned = {0: [], 1: [], 2: []}

    # ── Client C first (exhausts entire VP pool) ──────────────────────────
    c_dom   = client_dominance[2]["class"]   # VP = index 3
    vp_imgs = pools[c_dom][:]               # all VP training images
    assigned[2].extend(vp_imgs)
    pools[c_dom] = []                        # VP pool exhausted

    # Fill remaining slots to hit client_c_max
    c_remaining  = client_c_max - len(assigned[2])
    non_vp       = [l for l in range(n_classes) if l != c_dom]
    if c_remaining > 0:
        props = np.random.dirichlet([dirichlet_alpha] * len(non_vp))
        for i, label in enumerate(non_vp):
            n_take = min(int(round(props[i] * c_remaining)), len(pools[label]))
            assigned[2].extend(pools[label][:n_take])
            pools[label] = pools[label][n_take:]
    assigned[2] = assigned[2][:client_c_max]

    # ── Clients A and B — take ENTIRE dominant pool ────────────────────────
    for client_id in [0, 1]:
        dom   = client_dominance[client_id]["class"]
        ratio = client_dominance[client_id]["ratio"]

        # Take ALL remaining dominant class images
        dom_imgs  = pools[dom][:]
        assigned[client_id].extend(dom_imgs)
        pools[dom] = []

        # Fill non-dominant slots via Dirichlet to maintain ~70% ratio
        dom_count    = len(dom_imgs)
        total_target = int(dom_count / ratio)
        non_dom_need = total_target - dom_count

        other = [l for l in range(n_classes)
                 if l != dom and len(pools[l]) > 0]
        if other and non_dom_need > 0:
            props = np.random.dirichlet([dirichlet_alpha] * len(other))
            for i, label in enumerate(other):
                n_take = min(int(round(props[i] * non_dom_need)),
                             len(pools[label]))
                assigned[client_id].extend(pools[label][:n_take])
                pools[label] = pools[label][n_take:]

    # ── Distribute ALL leftover images — nothing wasted ───────────────────
    leftover = []
    for label in pools:
        leftover.extend(pools[label])

    if leftover:
        np.random.shuffle(leftover)
        # Split leftover proportionally by current client size
        sizes     = [len(assigned[c]) for c in range(3)]
        total_sz  = sum(sizes)
        shares    = [s / total_sz for s in sizes]
        start     = 0
        for cid in range(3):
            if cid == 2:
                chunk = leftover[start:]   # give remainder to last client
            else:
                n     = int(round(shares[cid] * len(leftover)))
                chunk = leftover[start:start + n]
                start += n
            assigned[cid].extend(chunk)
        print(f"  Distributed {len(leftover)} leftover images across clients.")

    return {k: np.array(v) for k, v in assigned.items()}

print("constrained_partition defined — uses ALL training images.")

In [ ]:
# ── CELL 9: Run partition ──────────────────────────────────────────────────
from collections import Counter

client_indices = constrained_partition(
    class_indices    = class_indices,
    client_dominance = CLIENT_DOMINANCE,
    client_c_max     = client_c_max,
    n_classes        = N_CLASSES,
    dirichlet_alpha  = DIRICHLET_ALPHA,
)

all_assigned = []
print("Partition summary:\n")
for cid, name in enumerate(client_names):
    idx    = client_indices[cid]
    labels = all_labels[idx]
    total  = len(idx)
    counts = Counter(labels.tolist())
    print(f"Client {name} — {total} images")
    for label, class_name in enumerate(CLASS_NAMES):
        n   = counts.get(label, 0)
        pct = 100 * n / total if total > 0 else 0
        dom = " ← dominant" if label == CLIENT_DOMINANCE[cid]["class"] else ""
        print(f"  {class_name:25s}: {n:5d}  ({pct:.1f}%){dom}")
    print()
    all_assigned.extend(idx.tolist())

# Verify no overlaps
assert len(all_assigned) == len(set(all_assigned)), \
    "OVERLAP DETECTED"

total_train = sum(len(train_indices[np.where(
    all_labels[train_indices] == l)[0]]) for l in range(N_CLASSES))
print(f"Total training images available : {total_train}")
print(f"Total assigned to clients       : {len(all_assigned)}")
print(f"Wasted                          : {total_train - len(all_assigned)}")
print(f"Overlap check                   : PASSED")

In [ ]:
# ── CELL 10: Save partition + test set ────────────────────────────────────
for cid, name in enumerate(client_names):
    idx = client_indices[cid]
    np.save(f"{CACHE_DIR}/client_{name}_indices.npy", idx)
    np.save(f"{CACHE_DIR}/client_{name}_labels.npy",  all_labels[idx])
    np.save(f"{CACHE_DIR}/client_{name}_paths.npy",   all_paths[idx])

np.save(f"{CACHE_DIR}/test_indices.npy", test_indices)
np.save(f"{CACHE_DIR}/test_labels.npy",  all_labels[test_indices])
np.save(f"{CACHE_DIR}/test_paths.npy",   all_paths[test_indices])

print("Partition saved to cache:")
for name in client_names:
    n = len(client_indices[client_names.index(name)])
    print(f"  client_{name}_*.npy  ({n} images)")
print(f"  test_*.npy           ({len(test_indices)} images)")

In [ ]:
# ── CELL 11: Load frozen MobileNetV2 backbone ─────────────────────────────
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import time

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

backbone = models.mobilenet_v2(pretrained=True)
backbone.classifier = nn.Identity()

for param in backbone.parameters():
    param.requires_grad = False

backbone.eval()
backbone.to(DEVICE)

# Verify output shape
with torch.no_grad():
    dummy = torch.zeros(1, 3, 224, 224).to(DEVICE)
    out   = backbone(dummy)
print(f"Backbone output shape : {out.shape}")  # expect (1, 1280)
print("Backbone frozen and ready.")

In [ ]:
# ── CELL 12: Transform + PathDataset ──────────────────────────────────────

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # CXR is grayscale → 3ch
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std= [0.229, 0.224, 0.225]
    ),
])

class PathDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img), self.labels[idx]


def extract_features(paths, labels, backbone, desc=""):
    dataset = PathDataset(paths, labels, transform)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)
    all_feats, all_lbls = [], []
    total = len(loader)

    for i, (imgs, lbls) in enumerate(loader):
        imgs = imgs.to(DEVICE)
        with torch.no_grad():
            feats = backbone(imgs)
        all_feats.append(feats.cpu().numpy())
        all_lbls.append(lbls.numpy())
        if (i + 1) % 20 == 0 or (i + 1) == total:
            print(f"  {desc}: {i+1}/{total} batches")

    import numpy as np
    return (np.concatenate(all_feats, axis=0),
            np.concatenate(all_lbls,  axis=0))

print("extract_features ready.")

In [ ]:
# ── CELL 13: Extract features for each client ─────────────────────────────
# Slow cell — ~5-10 min per client on CPU, ~1-2 min on GPU
# Saved immediately after each client so progress is not lost

for cid, name in enumerate(client_names):
    paths  = np.load(f"{CACHE_DIR}/client_{name}_paths.npy")
    labels = np.load(f"{CACHE_DIR}/client_{name}_labels.npy")

    print(f"\nClient {name}: {len(paths)} images")
    t0 = time.time()

    feats, lbls = extract_features(paths, labels, backbone, desc=f"Client {name}")
    elapsed     = time.time() - t0

    np.save(f"{CACHE_DIR}/client_{name}_features.npy",        feats)
    np.save(f"{CACHE_DIR}/client_{name}_labels_verified.npy", lbls)

    print(f"  Done in {elapsed:.1f}s")
    print(f"  Features : {feats.shape}")
    print(f"  Labels   : {lbls.shape}")
    print(f"  Saved    : client_{name}_features.npy")

In [ ]:
# ── CELL 14: Extract test set features ────────────────────────────────────

test_paths  = np.load(f"{CACHE_DIR}/test_paths.npy")
test_labels_arr = np.load(f"{CACHE_DIR}/test_labels.npy")

print(f"Test set: {len(test_paths)} images")
t0 = time.time()

test_feats, test_lbls = extract_features(
    test_paths, test_labels_arr, backbone, desc="Test")

elapsed = time.time() - t0
np.save(f"{CACHE_DIR}/test_features.npy",        test_feats)
np.save(f"{CACHE_DIR}/test_labels_verified.npy", test_lbls)

print(f"\nDone in {elapsed:.1f}s")
print(f"Features : {test_feats.shape}")
print(f"Saved    : test_features.npy")

In [ ]:
# ── CELL 15: Sanity check ─────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("Shape verification:")
for name in client_names:
    f = np.load(f"{CACHE_DIR}/client_{name}_features.npy")
    l = np.load(f"{CACHE_DIR}/client_{name}_labels_verified.npy")
    print(f"  Client {name}: {f.shape}  labels {l.shape}")

tf = np.load(f"{CACHE_DIR}/test_features.npy")
tl = np.load(f"{CACHE_DIR}/test_labels_verified.npy")
print(f"  Test set  : {tf.shape}  labels {tl.shape}")

print("\nLinear probe on Client A → test set:")
A_f = np.load(f"{CACHE_DIR}/client_A_features.npy")
A_l = np.load(f"{CACHE_DIR}/client_A_labels_verified.npy")

clf = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
clf.fit(A_f, A_l)
preds = clf.predict(tf)
acc   = accuracy_score(tl, preds)
print(f"  Accuracy: {acc:.3f}")

total_assigned = sum(
    len(np.load(f"{CACHE_DIR}/client_{n}_features.npy"))
    for n in client_names
)
print(f"\nTotal images in training : {total_assigned}")
print(f"Test images              : {len(tf)}")
print(f"Grand total used         : {total_assigned + len(tf)}")

if acc > 0.60:
    print("\nSanity check PASSED — ready for notebook 3.")
else:
    print("\nSanity check FAILED — check transform or backbone.")